In [ ]:
# --- PocketLM setup (works in Colab and locally) ---
try:
    import pocketlm  # already installed locally / in CI
except ImportError:
    # Colab: install straight from GitHub. The corpus ships *inside* the package,
    # so there is no repo to clone and no working directory to change.
    import subprocess
    import sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
                           "git+https://github.com/ni5h4nt/pocketlm"])
    import pocketlm  # noqa: F811

import os
import torch

DEVICE = "micro" if os.environ.get("POCKETLM_CI") else ("cuda" if torch.cuda.is_available() else "cpu")
CORPUS_PATH = pocketlm.corpus_path()   # packaged data: valid from any directory
print("device:", DEVICE, "(timing is guidance, not a contract)")

# l5.5 Over- and underfitting

Train on a deliberately tiny slice and watch the curves. **Overfitting**: train
loss keeps falling while val loss stalls or rises, the model is memorizing.
**Underfitting**: both stay high, the model lacks capacity or training.

In [ ]:
from pocketlm import (CharTokenizer, PocketLM, PocketLMConfig, init_weights,
                      make_batch, estimate_loss, split_data)
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

corpus = open(CORPUS_PATH, encoding="utf-8").read()
tok = CharTokenizer.from_text(corpus)
small = torch.tensor(tok.encode(corpus[:2000]), dtype=torch.long)  # tiny on purpose
train, val = split_data(small, 0.2)
torch.manual_seed(0)
cfg = PocketLMConfig(vocab_size=tok.vocab_size, block_size=32, n_layer=2,
                     n_head=2, n_embd=64)
model = init_weights(PocketLM(cfg))
opt = torch.optim.AdamW(model.parameters(), lr=1e-3)
g = torch.Generator().manual_seed(0)
steps = 80 if os.environ.get("POCKETLM_CI") else 500
tr_curve, va_curve = [], []
for s in range(steps):
    x, y = make_batch(train, 32, 16, generator=g)
    _, loss = model(x, y)
    opt.zero_grad(set_to_none=True)
    loss.backward()
    opt.step()
    if s % 10 == 0:
        tr_curve.append(estimate_loss(model, train, 32, 16, iters=5, generator=g))
        va_curve.append(estimate_loss(model, val, 32, 16, iters=5, generator=g))
plt.plot(tr_curve, label="train")
plt.plot(va_curve, label="val")
plt.legend()
plt.title("train vs val loss")
plt.show()
print("final train:", round(tr_curve[-1], 3), " val:", round(va_curve[-1], 3))

On a tiny dataset the train curve dives while val lags, the signature of
overfitting. The fixes: more data, fewer parameters, regularization, or early
stopping when val stops improving.

In [ ]:
assert tr_curve[-1] < tr_curve[0]   # the model did learn the train slice